# Build GeoPlan Agent Tools

This notebook wraps the compact outputs from the deterministic GIS
workflow as read-only LlamaIndex tools.

Run `01_site_selection.ipynb` first. The agent tools read exported
candidate summaries, the selected sequence, scenario tables, and
configuration files; they do not ask an LLM to perform spatial
calculations.


## 2. Imports and project-root discovery

Place `geoplan_agent_tools.py` beside this notebook, or copy it into your
project's `src/` folder.


In [15]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
SOURCE_DIR = PROJECT_ROOT / "src"

if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from geoplan_agent_tools import GeoPlanToolbox


## 3. Required deterministic outputs

The tools require three files produced by the final export section of the
main notebook:

```text
outputs/
├── agent_candidate_summary.json
├── selected_site_sequence.csv
└── planning_configuration.json
```

Geometry auditing also uses this optional but normally available file:

```text
data/processed/scored_ev_charging_candidates.geojson
```

Scenario and sensitivity tools use two optional exports:

```text
outputs/scenario_results.csv
outputs/sensitivity_results.csv
```


### Optional scenario and sensitivity exports

Add these two lines to the deterministic notebook after `scenario_table`
and `sensitivity_table` have been created:

```python
scenario_table.to_csv(
    OUTPUT_DIR / "scenario_results.csv",
    index=False,
)

sensitivity_table.to_csv(
    OUTPUT_DIR / "sensitivity_results.csv",
    index=False,
)
```

The core candidate and audit tools work without these optional files.


## 4. Load and validate the evidence store

Strict mode checks:

- required files exist;
- candidate IDs are unique;
- required score columns exist;
- scores lie within `[0, 1]`;
- selected IDs exist in the candidate summary;
- selected IDs are unique;
- category weights sum to one.


In [16]:
toolbox = GeoPlanToolbox(
    PROJECT_ROOT,
    strict=True,
)


## 5. Test the plain Python functions first

Tool logic should be verified before an LLM can call it. This separates data
bugs from agent-routing bugs.


In [17]:
configuration_result = (
    toolbox.get_planning_configuration()
)

print(
    json.dumps(
        configuration_result,
        indent=2,
    )
)


{
  "status": "ok",
  "configuration": {
    "city": "Toronto",
    "project_type": "public_ev_charging",
    "number_of_recommendations": 20,
    "analysis_crs": "EPSG:26917",
    "display_crs": "EPSG:4326",
    "distance_units": "metres",
    "buffer_distances_m": [
      500,
      1000,
      2000,
      5000
    ],
    "existing_charger_exclusion_m": 100,
    "minimum_selected_site_separation_m": 1500,
    "service_radius_m": 1000,
    "category_weights": {
      "demand": 0.3,
      "coverage": 0.25,
      "accessibility": 0.2,
      "feasibility": 0.15,
      "equity": 0.1
    }
  }
}


In [18]:
metrics_result = toolbox.list_available_metrics()

display(
    pd.DataFrame(
        metrics_result["numeric_metrics"]
    )
)


,metric,non_missing_count,preferred_direction
0,capacity_filled,212,higher
1,population_1000m,212,higher
2,traffic_volume_weighted_1000m,211,higher
3,distance_to_nearest_charger_m,212,higher
4,existing_chargers_2000m,212,lower
5,demand_score,212,higher
6,coverage_score,212,higher
7,accessibility_score,212,higher
8,feasibility_score,212,higher
9,equity_score,212,higher


In [19]:
candidate_list_result = toolbox.list_candidate_ids(
    limit=10
)

display(
    pd.DataFrame(
        candidate_list_result["candidates"]
    )
)

SAMPLE_CANDIDATE_ID = (
    candidate_list_result["candidates"][0][
        "candidate_id"
    ]
)

print("Sample candidate:", SAMPLE_CANDIDATE_ID)


,candidate_id,address,overall_score
0,green_p_710,100 Grangeway Avenue,0.787777
1,green_p_700,101 Grangeway Avenue,0.784461
2,green_p_821,Kennedy South Lot -155 Transway Cres,0.745421
3,green_p_706,284 Milner Avenue,0.742060
4,green_p_813,Finch West Lot - 18 Hendon Ave,0.732037
5,green_p_814,Finch East - 814 Willowdale Ave - partial clos...,0.723838
6,green_p_707,1530 Markham Road - minor capital repairs onsite,0.721630
7,green_p_826,Kennedy North Service Road - 2450 Eglinton Ave...,0.717570
8,green_p_711,158 Borough Drive,0.716935
9,green_p_818,Lawrence East Lot - Near 2450 Lawrence Ave East,0.707290


Sample candidate: green_p_710


In [20]:
candidate_result = toolbox.get_candidate(
    SAMPLE_CANDIDATE_ID
)

print(
    json.dumps(
        candidate_result,
        indent=2,
    )
)


{
  "status": "ok",
  "candidate": {
    "candidate_id": "green_p_710",
    "address": "100 Grangeway Avenue",
    "capacity_filled": 214,
    "population_1000m": 17841.6782426689,
    "traffic_volume_weighted_1000m": 17514.4533913057,
    "distance_to_nearest_charger_m": 10033.7017143517,
    "existing_chargers_2000m": 0,
    "demand_score": 0.8238538964,
    "coverage_score": 0.9189174834,
    "accessibility_score": 0.6427145009,
    "feasibility_score": 0.6364746356,
    "equity_score": 0.8687760823,
    "overall_score": 0.7877772435,
    "traffic_data_quality": "high",
    "capacity_missing": false
  }
}


In [21]:
top_result = toolbox.get_top_candidates(
    metric="overall_score",
    n=10,
    order="desc",
)

display(
    pd.DataFrame(
        top_result["candidates"]
    )
)


,candidate_id,address,overall_score,demand_score,coverage_score,accessibility_score,feasibility_score,equity_score,traffic_data_quality,capacity_missing
0,green_p_710,100 Grangeway Avenue,0.787777,0.823854,0.918917,0.642715,0.636475,0.868776,high,False
1,green_p_700,101 Grangeway Avenue,0.784461,0.808955,0.925644,0.622128,0.659691,0.869841,high,False
2,green_p_821,Kennedy South Lot -155 Transway Cres,0.745421,0.790438,0.665643,0.645437,0.870646,0.821949,high,False
3,green_p_706,284 Milner Avenue,0.742060,0.721455,0.997571,0.619992,0.551194,0.695531,high,False
4,green_p_813,Finch West Lot - 18 Hendon Ave,0.732037,0.872763,0.427654,0.638109,0.968663,0.903729,high,False
5,green_p_814,Finch East - 814 Willowdale Ave - partial clos...,0.723838,0.872001,0.429340,0.643923,0.904732,0.904085,high,False
6,green_p_707,1530 Markham Road - minor capital repairs onsite,0.721630,0.725176,1.000000,0.635005,0.383800,0.695060,high,False
7,green_p_826,Kennedy North Service Road - 2450 Eglinton Ave...,0.717570,0.794754,0.678480,0.591081,0.727982,0.821104,high,False
8,green_p_711,158 Borough Drive,0.716935,0.787927,0.892873,0.620616,0.338513,0.824383,high,False
9,green_p_818,Lawrence East Lot - Near 2450 Lawrence Ave East,0.707290,0.783931,0.772311,0.532921,0.635514,0.771215,medium,False


In [22]:
comparison_ids = [
    record["candidate_id"]
    for record
    in candidate_list_result["candidates"][:3]
]

comparison_result = toolbox.compare_candidates(
    comparison_ids
)

display(
    pd.DataFrame(
        comparison_result["records"]
    )
)

print(
    json.dumps(
        comparison_result["best_by_metric"],
        indent=2,
    )
)


,candidate_id,address,overall_score,demand_score,coverage_score,accessibility_score,feasibility_score,equity_score,capacity_filled,population_1000m,traffic_volume_weighted_1000m,distance_to_nearest_charger_m,existing_chargers_2000m,traffic_data_quality,capacity_missing
0,green_p_710,100 Grangeway Avenue,0.787777,0.823854,0.918917,0.642715,0.636475,0.868776,214,17841.678243,17514.453391,10033.701714,0,high,False
1,green_p_700,101 Grangeway Avenue,0.784461,0.808955,0.925644,0.622128,0.659691,0.869841,261,16822.653374,15298.341047,10141.587715,0,high,False
2,green_p_821,Kennedy South Lot -155 Transway Cres,0.745421,0.790438,0.665643,0.645437,0.870646,0.821949,673,22184.985754,14222.914666,5971.341673,0,high,False


{
  "overall_score": {
    "candidate_id": "green_p_710",
    "value": 0.7877772435,
    "preferred_direction": "higher"
  },
  "demand_score": {
    "candidate_id": "green_p_710",
    "value": 0.8238538964,
    "preferred_direction": "higher"
  },
  "coverage_score": {
    "candidate_id": "green_p_700",
    "value": 0.9256438201,
    "preferred_direction": "higher"
  },
  "accessibility_score": {
    "candidate_id": "green_p_821",
    "value": 0.6454366541,
    "preferred_direction": "higher"
  },
  "feasibility_score": {
    "candidate_id": "green_p_821",
    "value": 0.870645555,
    "preferred_direction": "higher"
  },
  "equity_score": {
    "candidate_id": "green_p_700",
    "value": 0.8698412476,
    "preferred_direction": "higher"
  },
  "capacity_filled": {
    "candidate_id": "green_p_821",
    "value": 673,
    "preferred_direction": "higher"
  },
  "population_1000m": {
    "candidate_id": "green_p_821",
    "value": 22184.9857539455,
    "preferred_direction": "higher"
  }

In [23]:
selected_result = toolbox.get_selected_sequence(
    n=20
)

if selected_result["status"] == "ok":
    display(
        pd.DataFrame(
            selected_result["selection"]
        )
    )
else:
    print(selected_result)


,selection_round,candidate_id,address,selection_score,marginal_population,capacity_filled,_distance,overall_score
0,1,green_p_710,100 Grangeway Avenue,0.730789,17892.0,214,10033.701714,0.787777
1,2,green_p_821,Kennedy South Lot -155 Transway Cres,0.783931,25262.0,673,4846.685332,0.745421
2,3,green_p_813,Finch West Lot - 18 Hendon Ave,0.773663,29868.0,1552,2154.160920,0.732037
3,4,green_p_816,Don Mills Lot - 1800 Sheppard Ave East,0.788001,28198.0,366,2471.944273,0.700234
4,5,green_p_802,Islington Main Lot - 3330 Bloor Street West,0.740731,21146.0,534,1835.686460,0.681206
5,6,green_p_811,Wilson Main Lot - 50 Wilson Heights Blvd,0.712686,8997.0,885,4069.043847,0.690289
6,7,green_p_668,2700 Eglinton Avenue West,0.725413,15510.0,109,3022.437315,0.625249
7,8,green_p_823,Warden South Lot - 701 Warden Ave,0.725030,16273.0,151,2756.799587,0.642166
8,9,green_p_663,1 Shortt Street,0.717633,20889.0,130,1997.383130,0.607477
9,10,green_p_824,Victoria Main Lot - 777 Victoria Park Avenue,0.710511,21682.0,173,1260.257553,0.637251


## 6. Test deterministic auditing

The candidate audit flags:

- imputed parking capacity;
- low or unavailable traffic-data quality;
- missing evidence;
- scores outside `[0, 1]`;
- capacity below the prototype threshold;
- conflict with the known-charger exclusion distance.

The recommendation-set audit additionally checks:

- unknown IDs;
- duplicate IDs;
- too many recommendations;
- candidate-level warnings;
- minimum pairwise spacing when the scored GeoJSON is available.


In [24]:
candidate_audit = toolbox.audit_candidate(
    SAMPLE_CANDIDATE_ID
)

print(
    json.dumps(
        candidate_audit,
        indent=2,
    )
)


{
  "status": "ok",
  "candidate_id": "green_p_710",
  "risk_level": "low",
  "critical_issues": [],
  "warnings": [],
  "approved_for_planning_screening": true
}


In [25]:
recommendation_audit = (
    toolbox.audit_recommendation_set()
)

print(
    json.dumps(
        recommendation_audit,
        indent=2,
    )
)


{
  "status": "ok",
  "candidate_count": 20,
  "candidate_ids": [
    "green_p_710",
    "green_p_821",
    "green_p_813",
    "green_p_816",
    "green_p_802",
    "green_p_811",
    "green_p_668",
    "green_p_823",
    "green_p_663",
    "green_p_824",
    "green_p_152",
    "green_p_709",
    "green_p_510",
    "green_p_818",
    "green_p_602",
    "green_p_283",
    "green_p_266",
    "green_p_657",
    "green_p_706",
    "green_p_47"
  ],
  "critical_issues": [],
  "warnings": [],
  "candidate_audits": [
    {
      "status": "ok",
      "candidate_id": "green_p_710",
      "risk_level": "low",
      "critical_issues": [],
      "warnings": [],
      "approved_for_planning_screening": true
    },
    {
      "status": "ok",
      "candidate_id": "green_p_821",
      "risk_level": "low",
      "critical_issues": [],
      "warnings": [],
      "approved_for_planning_screening": true
    },
    {
      "status": "ok",
      "candidate_id": "green_p_813",
      "risk_level": "low",


## 7. Test optional scenario and sensitivity tools


In [26]:
scenario_status = toolbox.get_scenario_results()
sensitivity_status = (
    toolbox.get_sensitivity_results(n=10)
)

print("Scenario tool:")
print(
    json.dumps(
        scenario_status,
        indent=2,
    )
)

print("\nSensitivity tool:")
print(
    json.dumps(
        sensitivity_status,
        indent=2,
    )
)


Scenario tool:
{
  "status": "ok",
  "available_scenarios": [
    "balanced",
    "coverage_first",
    "demand_first",
    "equity_balanced"
  ],
  "row_count": 80
}

Sensitivity tool:
{
  "status": "ok",
  "minimum_frequency": 0.0,
  "returned": 10,
  "candidates": [
    {
      "candidate_id": "green_p_663",
      "times_selected": 100,
      "selection_frequency": 1.0,
      "address": "1 Shortt Street",
      "overall_score": 0.6074770835736472
    },
    {
      "candidate_id": "green_p_816",
      "times_selected": 100,
      "selection_frequency": 1.0,
      "address": "Don Mills Lot - 1800 Sheppard Ave East",
      "overall_score": 0.7002337786544738
    },
    {
      "candidate_id": "green_p_668",
      "times_selected": 100,
      "selection_frequency": 1.0,
      "address": "2700 Eglinton Avenue West",
      "overall_score": 0.625248621230211
    },
    {
      "candidate_id": "green_p_206",
      "times_selected": 100,
      "selection_frequency": 1.0,
      "address": "2

## 8. Wrap the functions as LlamaIndex `FunctionTool` objects

`FunctionTool.from_defaults` uses the Python function signature, type
annotations, name, and docstring to construct the schema and description
shown to an agent.

No LLM is invoked in this section.


In [28]:
tools = (
    toolbox.create_llamaindex_tools()
)

print(
    "Created tools:",
    len(tools),
)

tool_metadata = pd.DataFrame(
    [
        {
            "tool_name": name,
            "description": (
                tool.metadata.description
            ),
        }
        for name, tool
        in tools.items()
    ]
)

display(tool_metadata)


Created tools: 11


,tool_name,description
0,get_planning_configuration,"get_planning_configuration() -> dict[str, typi..."
1,list_available_metrics,"list_available_metrics() -> dict[str, typing.A..."
2,list_candidate_ids,list_candidate_ids(limit: int = 50) -> dict[st...
3,get_candidate,"get_candidate(candidate_id: str) -> dict[str, ..."
4,get_top_candidates,get_top_candidates(metric: str = 'overall_scor...
5,compare_candidates,"compare_candidates(candidate_ids: list[str], m..."
6,get_selected_sequence,get_selected_sequence(n: int = 50) -> dict[str...
7,get_scenario_results,get_scenario_results(scenario_name: str | None...
8,get_sensitivity_results,"get_sensitivity_results(n: int = 20, minimum_f..."
9,audit_candidate,audit_candidate(candidate_id: str) -> dict[str...


## 9. Organize tools by specialist-agent role

Restricting each agent to relevant tools makes the workflow easier to
understand and evaluate.


In [29]:
role_tool_names = toolbox.role_tool_names()
role_tool_groups = (
    toolbox.create_role_tool_groups()
)

display(
    pd.DataFrame(
        [
            {
                "agent_role": role,
                "tool_count": len(tool_names),
                "tools": ", ".join(tool_names),
            }
            for role, tool_names
            in role_tool_names.items()
        ]
    )
)


,agent_role,tool_count,tools
0,PlanningCoordinator,4,"get_planning_configuration, list_candidate_ids..."
1,DemandCoverageAgent,4,"get_candidate, get_top_candidates, compare_can..."
2,AccessibilityFeasibilityAgent,4,"get_candidate, get_top_candidates, compare_can..."
3,EquityRiskAgent,4,"get_candidate, compare_candidates, audit_candi..."
4,ScenarioAnalysisAgent,3,"get_scenario_results, get_sensitivity_results,..."
5,FinalReviewerAgent,5,"get_planning_configuration, get_candidate, get..."


### Proposed role assignments

**PlanningCoordinator**

Reads configuration and the selected sequence, then delegates specialist
analyses and requests a final audit.

**DemandCoverageAgent**

Retrieves candidate evidence, rankings, comparisons, and sequential
selections.

**AccessibilityFeasibilityAgent**

Examines accessibility and feasibility fields and audits data limitations.

**EquityRiskAgent**

Reviews equity evidence, quality flags, and recommendation-set risks.

**ScenarioAnalysisAgent**

Reads planning-scenario and weight-sensitivity exports.

**FinalReviewerAgent**

performs deterministic audits before approving a final planning report.


## 10. Save a tool manifest

The manifest records available tools, recommended role assignments, and
evidence-file status. It can be included in the project outputs for
reproducibility.


In [30]:
manifest_path = toolbox.save_tool_manifest()

print("Saved manifest:", manifest_path.resolve())

with manifest_path.open(
    "r",
    encoding="utf-8",
) as file:
    manifest = json.load(file)

print(
    json.dumps(
        manifest["role_tool_names"],
        indent=2,
    )
)


Saved manifest: /Users/miladsaeedi/Desktop/Daily_Work_load/Projects_for_CV/Geospatial_site_selection/outputs/agent_tool_manifest.json
{
  "PlanningCoordinator": [
    "get_planning_configuration",
    "list_candidate_ids",
    "get_selected_sequence",
    "audit_recommendation_set"
  ],
  "DemandCoverageAgent": [
    "get_candidate",
    "get_top_candidates",
    "compare_candidates",
    "get_selected_sequence"
  ],
  "AccessibilityFeasibilityAgent": [
    "get_candidate",
    "get_top_candidates",
    "compare_candidates",
    "audit_candidate"
  ],
  "EquityRiskAgent": [
    "get_candidate",
    "compare_candidates",
    "audit_candidate",
    "audit_recommendation_set"
  ],
  "ScenarioAnalysisAgent": [
    "get_scenario_results",
    "get_sensitivity_results",
    "compare_candidates"
  ],
  "FinalReviewerAgent": [
    "get_planning_configuration",
    "get_candidate",
    "get_selected_sequence",
    "audit_candidate",
    "audit_recommendation_set"
  ]
}


## 11. Minimal regression tests

These assertions catch common failures before the tools are connected to
agents.


In [31]:
assert len(toolbox.candidates) > 0
assert toolbox.candidates["candidate_id"].is_unique

assert toolbox.get_candidate(
    SAMPLE_CANDIDATE_ID
)["status"] == "ok"

assert toolbox.get_candidate(
    "not_a_real_candidate"
)["status"] == "error"

assert toolbox.get_top_candidates(
    "overall_score",
    n=5,
)["returned"] <= 5

assert toolbox.compare_candidates(
    comparison_ids
)["status"] == "ok"

assert (
    toolbox.get_selected_sequence(
        n=20
    )["status"]
    == "ok"
)

assert (
    toolbox.audit_recommendation_set()[
        "status"
    ]
    == "ok"
)

assert set(role_tool_names).issuperset(
    {
        "PlanningCoordinator",
        "DemandCoverageAgent",
        "AccessibilityFeasibilityAgent",
        "EquityRiskAgent",
        "ScenarioAnalysisAgent",
        "FinalReviewerAgent",
    }
)

print("All deterministic tool tests passed.")


All deterministic tool tests passed.


# Tool layer complete

The following runtime objects are now ready for the multi-agent notebook:

```python
toolbox
llamaindex_tools
role_tool_names
role_tool_groups
```

The next notebook should:

1. configure a local or hosted LLM;
2. define structured Pydantic response schemas;
3. create the six specialist agents;
4. construct a LlamaIndex `AgentWorkflow`;
5. stream tool calls and agent handoffs;
6. evaluate routing, tool selection, evidence accuracy, and review outcomes.
